# Fine-Tuning Whisper-Tiny on MInds14 (English) — Google Colab, Submission-Ready

This notebook fine-tunes `openai/whisper-tiny` on the `PolyAI/minds14` (`en-US`) dataset and pushes the result to the Hugging Face Hub in the format the **Hugging Face Audio Course, Unit 5** grader expects.




## 0. Install dependencies

Colab comes with an older/incompatible set of these libraries pre-installed (or missing entirely), so install fresh versions first.

In [ ]:
!pip install --upgrade -q datasets transformers evaluate accelerate jiwer soundfile librosa huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 55.9 MB/s eta 0:00:00


## 1. Load the dataset



In [ ]:
from datasets import load_dataset

minds = load_dataset("PolyAI/minds14", "en-US", split="train")


README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

en-US/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 34.2MB            

en-US/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/563 [00:00<?, ? examples/s]

## 2. Keep only the columns we need


In [ ]:
minds = minds.select_columns(["audio", "transcription"])
minds = minds.rename_column("transcription", "sentence")

## 3. Load the Whisper processor

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-tiny", language="english", task="transcribe"
)


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

## 4. Resample audio to match the processor's expected sampling rate

In [ ]:
from datasets import Audio

sampling_rate = processor.feature_extractor.sampling_rate
minds = minds.cast_column("audio", Audio(sampling_rate=sampling_rate))


## 5. Extract audio features and tokenize transcripts

Maps every example to `input_features` (log-mel spectrogram), `labels` (tokenized transcript), and `input_length` (audio duration in seconds, needed by the filtering step next).

In [ ]:
def prepare_dataset(example):
    audio = example["audio"]

    example = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],
        text=example["sentence"],
    )

    # compute input length of audio sample in seconds
    example["input_length"] = len(audio["array"]) / audio["sampling_rate"]

    return example


minds = minds.map(
    prepare_dataset, num_proc=1
)


Map (num_proc=1):   0%|          | 0/563 [00:00<?, ? examples/s]

## 6. Filter out audio clips longer than 30 seconds


In [ ]:
max_input_length = 30.0


def is_audio_in_length_range(length):
    return length < max_input_length


minds = minds.filter(
    is_audio_in_length_range,
    input_columns=["input_length"],
)


Filter:   0%|          | 0/563 [00:00<?, ? examples/s]

## 7. Split into train / evaluation sets



In [ ]:
train_dataset = minds.select(range(450))
evaluation_dataset = minds.select(range(450, len(minds)))


## 8. Load the model, freeze the encoder, and fix the generation config

The plan is to freeze the encoder entirely, except for the cross-attention (`encoder_attn`) layers inside the decoder, which stay trainable.


In [ ]:
import torch
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")

for name, param in model.named_parameters():
    if "encoder_attn" in name:
        param.requires_grad = True
    elif "encoder" in name:
        param.requires_grad = False
    else:
        param.requires_grad = False

# Make sure generation actually targets English transcription
model.generation_config.language = "english"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None


model.safetensors: reconstructing file:   0%|          |  0.00B /  151MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.75k [00:00<?, ?B/s]

## 9. Custom data collator

Pads audio features and label sequences to the longest item in each batch, and masks label padding with `-100` so it's ignored by the loss.


In [ ]:
import torch

# 1. Create a custom collator wrapper to squeeze out the extra dimension
class SpeechDataCollatorWithPadding:
    def __init__(self, base_collator):
        self.base_collator = base_collator

    def __call__(self, features):
        # Call your original data collator first
        batch = self.base_collator(features)

        # If input_features is 4D, squeeze it to 3D [8, 80, 3000]
        if "input_features" in batch and batch["input_features"].ndim == 4:
            batch["input_features"] = batch["input_features"].squeeze(1)

        return batch

# 2. Wrap your original data_collator (assuming it was named 'data_collator')
wrapped_data_collator = SpeechDataCollatorWithPadding(data_collator)


## 10. Load the WER metric

In [ ]:
import evaluate

metric = evaluate.load("wer")


## 11. Compute metrics function

Computes both the raw ("orthographic") WER and a normalized WER using Whisper's `BasicTextNormalizer`, filtering out empty references.

In [ ]:
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

normalizer = BasicTextNormalizer()


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    # compute orthographic wer
    wer_ortho = metric.compute(predictions=pred_str, references=label_str)

    # compute normalised WER
    pred_str_norm = [normalizer(pred) for pred in pred_str]
    label_str_norm = [normalizer(label) for label in label_str]
    # only evaluate samples that correspond to non-zero references
    pred_str_norm = [
        pred_str_norm[i] for i in range(len(pred_str_norm)) if len(label_str_norm[i]) > 0
    ]
    label_str_norm = [
        label_str_norm[i]
        for i in range(len(label_str_norm))
        if len(label_str_norm[i]) > 0
    ]

    wer = metric.compute(predictions=pred_str_norm, references=label_str_norm)

    return {"wer_ortho": wer_ortho, "wer": wer}


## 12. Check that a GPU is available

The `Trainer` will automatically move the model and batches to a GPU if one is visible to PyTorch. Run this to confirm before training — `fp16=True` below will fail on CPU.

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected - fp16=True in the training arguments will fail on CPU.")
    print("If on Colab: Runtime > Change runtime type > Hardware accelerator > GPU")
    print("If on Kaggle: Settings > Accelerator > GPU")


CUDA available: True
Device name: Tesla T4


## 13. Log in to the Hugging Face Hub

You need a token with **write** access from https://huggingface.co/settings/tokens.

**Moved up (this pass):** this used to happen *after* the training-arguments/Trainer cell below, but `push_to_hub=True` needs you authenticated at the moment the `Trainer` is constructed, not just when you eventually call `push_to_hub()`.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()


## 14. Training arguments


In [ ]:
from transformers import Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    # 1. Output & Hub
    output_dir="./whisper-finetuned",
    push_to_hub=True,
    hub_model_id="YOUR_HF_USERNAME/whisper-tiny-minds14",  # <-- replace with your HF username

    # 2. Batch sizes & steps
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    num_train_epochs=4,

    # 3. Learning rate & optimizer
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,

    # 4. Hardware optimization
    fp16=True,                     # half precision training
    gradient_checkpointing=True,   # saves VRAM at the cost of slight compute

    # 5. Generation during evaluation (required for compute_metrics to get real predictions)
    predict_with_generate=True,
    generation_max_length=225,

    # 6. Evaluation, logging & checkpoint selection
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to=["tensorboard"],
)


## 15. Save the processor alongside the model

The grader loads your repo with `pipeline("automatic-speech-recognition", model=...)`, which needs the feature extractor + tokenizer (i.e. the full `WhisperProcessor`), not just the tokenizer. Saving it into `output_dir` now means it gets picked up and pushed together with the model checkpoints.

In [ ]:
processor.save_pretrained(args.output_dir)


['./whisper-finetuned/processor_config.json']

## 16. Build the Trainer


In [ ]:
import os
from huggingface_hub import login
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    AutoModelForSeq2SeqLM,
    AutoProcessor
)

# 1. Authenticate with Hugging Face using your write token
login(token="")

# 2. Define your training arguments
# REPLACE "YOUR_REAL_HF_USERNAME" with your exact Hugging Face username below:
MY_USERNAME = "Atrac"

args = Seq2SeqTrainingArguments(
    output_dir=f"{MY_USERNAME}/my-seq2seq-model",
    hub_model_id=f"{MY_USERNAME}/my-seq2seq-model",
    push_to_hub=True,
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=40,
    predict_with_generate=True,
    fp16=True,
    report_to="none"
)

# 3. Initialize the trainer
trainer = Seq2SeqTrainer(
    args=args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=evaluation_dataset,
    data_collator=wrapped_data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)



In [ ]:
# 4. Start the training process
trainer.train()


Epoch,Training Loss,Validation Loss,Wer Ortho,Wer
1,No log,0.598185,0.354569,0.353629
2,No log,0.624370,0.381596,0.383764
3,No log,0.623969,0.366152,0.362854
4,No log,0.632075,0.342342,0.342558
5,No log,0.652166,0.373230,0.371464
6,No log,0.658162,0.352638,0.352399
7,No log,0.662802,0.350708,0.350554
8,No log,0.668308,0.350064,0.349938
9,0.020332,0.673444,0.344273,0.344403
10,0.020332,0.680408,0.345560,0.345633


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2280, training_loss=0.005640740836398643, metrics={'train_runtime': 3107.435, 'train_samples_per_second': 5.793, 'train_steps_per_second': 0.734, 'total_flos': 4.4313993216e+17, 'train_loss': 0.005640740836398643, 'epoch': 40.0})

##push the final model with the required kwargs
These are the exact `kwargs` the course requires for your result to count toward the certificate. `trainer.push_to_hub(**kwargs)` uploads the model weights and builds a model card tagged with this metadata, so the course's grading pipeline can find and evaluate it. Your **normalised WER** (the `"wer"` value, not `"wer_ortho"`) needs to come out **below 0.37** to pass.


In [ ]:

kwargs = {
    "dataset_tags": "PolyAI/minds14",
    "finetuned_from": "openai/whisper-tiny",
    "tasks": "automatic-speech-recognition",
}

trainer.push_to_hub(**kwargs)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/Atrac/my-seq2seq-model/commit/be76f0d7810fc2231179666b4e4f74982a6849b9', commit_message='End of training', commit_description='', oid='be76f0d7810fc2231179666b4e4f74982a6849b9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Atrac/my-seq2seq-model', endpoint='https://huggingface.co', repo_type='model', repo_id='Atrac/my-seq2seq-model'), pr_revision=None, pr_num=None)

## 18. Sanity-check the uploaded model

Reload it from the Hub the same way the grader will, print the eval metrics, and actually run the pipeline on one real sample so you can see it working end-to-end.



In [ ]:
from transformers import pipeline

hub_model_id = args.hub_model_id  # same repo you pushed to above
pipe = pipeline("automatic-speech-recognition", model=hub_model_id)

metrics = trainer.evaluate()
print(metrics)
print("Normalised WER:", metrics["eval_wer"])
print("Passing threshold: < 0.37")

# Run the reloaded, pushed model on one real audio sample as an end-to-end check
sample = minds[0]
result = pipe(sample["audio"]["array"])
print("\nModel transcription:", result["text"])
print("Ground truth:        ", sample["sentence"])


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Wer Ortho,Wer
0.204350,0.583932,10,0.383526,0.381919


{'eval_loss': 0.5839320421218872, 'eval_wer_ortho': 0.3835263835263835, 'eval_wer': 0.38191881918819187}
Normalised WER: 0.38191881918819187
Passing threshold: < 0.37

Model transcription: I would like to set up a joint account with my partner
Ground truth:         I would like to set up a joint account with my partner
